# Financial Market Analyst Agent

Automated analysis pipeline for Federal Reserve Staff Reports using:
- **LandingAI's ADE** for document extraction
- **AWS S3** for data storage
- **AWS Bedrock Knowledge Base** for semantic search
- **Strands Agent Framework** with Claude Sonnet 4.0

## 1. Setup and Configuration

In [18]:
import os
import json
import boto3
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

LANDINGAI_API_KEY = os.getenv('LANDINGAI_API_KEY')
AWS_REGION = os.getenv('AWS_REGION', 'us-west-2')
S3_BUCKET_NAME = os.getenv('S3_BUCKET_NAME')
BEDROCK_DATA_SOURCE_ID = os.getenv('BEDROCK_DATA_SOURCE_ID')
BEDROCK_MODEL_ID = os.getenv('BEDROCK_MODEL_ID', 'us.anthropic.claude-sonnet-4-5-20250929-v1:0')
BEDROCK_KB_ID = os.getenv('BEDROCK_KB_ID')

print("✓ Configuration loaded")
print(f"  - AWS Region: {AWS_REGION}")
print(f"  - S3 Bucket: {S3_BUCKET_NAME}")
print(f"  - Model: {BEDROCK_MODEL_ID}")

✓ Configuration loaded
  - AWS Region: us-west-2
  - S3 Bucket: lai-bedrock-strands-agentcore
  - Model: us.anthropic.claude-sonnet-4-20250514-v1:0


## 2. Load Extracted Document Data

In [6]:
# Load from markdown or JSON file
markdown_file = 'Credit-Card-Banking-Federal-Reserve.extraction.md'
json_file = 'Credit-Card-Banking-Federal-Reserve.extraction.json'

if os.path.exists(markdown_file):
    print(f"Loading from: {markdown_file}")
    with open(markdown_file, 'r') as f:
        markdown_content = f.read()
    extracted_data = {
        'result': {
            'markdown': markdown_content
        }
    }
    print(f"✓ Loaded {len(markdown_content)} characters")
elif os.path.exists(json_file):
    print(f"Loading from: {json_file}")
    with open(json_file, 'r') as f:
        extracted_data = json.load(f)
    print(f"✓ Loaded extraction result")
else:
    raise FileNotFoundError(f"Please place '{markdown_file}' or '{json_file}' in this directory")

Loading from: Credit-Card-Banking-Federal-Reserve.extraction.md
✓ Loaded 199094 characters


## 3. Upload to S3

In [ ]:
s3_client = boto3.client('s3', region_name=AWS_REGION)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
s3_key = f"financial-reports/fed-credit-card-banking_{timestamp}.json"

s3_client.put_object(
    Bucket=S3_BUCKET_NAME,
    Key=s3_key,
    Body=json.dumps(extracted_data, indent=2),
    ContentType='application/json'
)

s3_uri = f"s3://{S3_BUCKET_NAME}/{s3_key}"
print(f"✓ Data uploaded to: {s3_uri}")

## 4. Sync Bedrock Knowledge Base

In [ ]:
if BEDROCK_KB_ID:
    bedrock_agent = boto3.client('bedrock-agent', region_name=AWS_REGION)
    
    response = bedrock_agent.start_ingestion_job(
        knowledgeBaseId=BEDROCK_KB_ID,
        dataSourceId=BEDROCK_DATA_SOURCE_ID
    )
    
    print(f"✓ Knowledge base sync initiated")
    print(f"  - Job ID: {response.get('ingestionJob', {}).get('ingestionJobId')}")
else:
    print("⚠ BEDROCK_KB_ID not set - skipping KB sync")

## 5. Create Financial Analyst Tools

In [19]:
import strands

@strands.tool
def search_knowledge_base(query: str, max_results: int = 5) -> dict:
    """
    Search the Bedrock Knowledge Base for relevant information about Federal Reserve credit card banking.
    
    Use this tool to find:
    - Interest rate trends and data
    - Market conditions and analysis
    - Policy impacts and regulatory information
    - Financial metrics and statistics
    - Historical comparisons and trends
    
    Args:
        query: Natural language question or search query
        max_results: Maximum number of results to return (default: 5)
    
    Returns:
        Relevant passages from the Federal Reserve report with context
    """
    if not BEDROCK_KB_ID:
        return {'error': 'Knowledge base not configured. Please set BEDROCK_KB_ID in .env file.'}
    
    try:
        bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=AWS_REGION)
        
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=BEDROCK_KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': max_results
                }
            }
        )
        
        results = []
        for item in response.get('retrievalResults', []):
            results.append({
                'content': item.get('content', {}).get('text', ''),
                'score': item.get('score', 0),
                'location': item.get('location', {})
            })
        
        return {
            'query': query,
            'results_count': len(results),
            'results': results
        }
    
    except Exception as e:
        return {
            'error': f'Knowledge base search failed: {str(e)}',
            'query': query
        }

print("✓ Knowledge Base search tool created")

✓ Knowledge Base search tool created


## 6. Initialize Strands Agent

In [12]:
from strands import Agent

agent = Agent(
    model=BEDROCK_MODEL_ID,
    name="Financial Market Analyst",
    description="Expert agent for analyzing Federal Reserve credit card banking reports",
    system_prompt="""
You are a Financial Market Analyst specializing in credit card banking and Federal Reserve reports.

When responding:
- If a question is ambiguous, ask clarifying questions before providing analysis
- If you need specific parameters (time periods, metrics, segments), ask the user to specify
- Use your tools to gather data, then provide clear, data-driven insights
- Be conversational and interactive

You have access to structured data extracted from Federal Reserve credit card banking reports.
Provide clear, data-driven insights backed by the available financial data.
""",
    tools=[search_knowledge_base]
)

print("✓ Agent initialized")
print(f"  - Model: {BEDROCK_MODEL_ID}")
print(f"  - Tools: {len(agent.tool_names)}")

✓ Agent initialized
  - Model: us.anthropic.claude-sonnet-4-20250514-v1:0
  - Tools: 1


## 7. Interactive Chat Loop

In [ ]:
print("="*70)
print("Financial Market Analyst Agent - Interactive Chat")
print("="*70)
print("\nAsk questions about the Federal Reserve credit card banking report.")
print("Type 'exit', 'quit', or 'bye' to end the conversation.")
print("="*70 + "\n")

while True:
    try:
        user_input = input("\n🧑 You: ").strip()
        
        if not user_input:
            continue
            
        if user_input.lower() in ['exit', 'quit', 'bye', 'q']:
            print("\n👋 Ending conversation. Goodbye!")
            break
        
        print("\n🤖 Agent: ", end="")
        result = agent(user_input)
        print(result)
        
    except KeyboardInterrupt:
        print("\n\n👋 Conversation interrupted. Goodbye!")
        break
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Please try again or type 'exit' to quit.")

Financial Market Analyst Agent - Interactive Chat

Ask questions about the Federal Reserve credit card banking report.
Type 'exit', 'quit', or 'bye' to end the conversation.




🧑 You:  What is the relationship between FICO scores, interest income, credit losses, and default-adjusted credit spreads according to Figure 4 of the Fed report? Page 49



🤖 Agent: I'll search for information about Figure 4 from page 49 of the Fed report, which discusses the relationship between FICO scores, interest income, credit losses, and default-adjusted credit spreads.
Tool #1: search_knowledge_base
I apologize, but I'm currently experiencing a technical issue accessing the knowledge base. Let me try a different search approach to find information about the relationship between FICO scores and credit metrics from the Federal Reserve report.
Tool #2: search_knowledge_base
I'm experiencing a technical issue with accessing the knowledge base at the moment. The search function is returning an expired token error, which prevents me from retrieving the specific information about Figure 4 on page 49 of the Fed report.

Without access to the specific figure and data, I cannot provide the exact details about the relationship between FICO scores, interest income, credit losses, and default-adjusted credit spreads as shown in that particular figure.

Could 


🧑 You:  What is the relationship between FICO scores, interest income, credit losses, and default-adjusted credit spreads according to Figure 4 of the Fed report? Page 49



🤖 Agent: Let me try to access the knowledge base again to find information about Figure 4 from page 49 regarding the relationship between FICO scores and credit metrics.
Tool #3: search_knowledge_base

Tool #4: search_knowledge_base
I'm still encountering the same technical issue with the knowledge base access. The system appears to have an expired authentication token that's preventing me from retrieving the specific information about Figure 4 from page 49 of the Federal Reserve report.

This seems to be a persistent technical issue rather than a temporary glitch. Without being able to access the knowledge base, I cannot provide you with the specific details about the relationship between FICO scores, interest income, credit losses, and default-adjusted credit spreads as presented in Figure 4.

To get this information, you may need to:
1. Contact technical support about the knowledge base access issue
2. Consult the Federal Reserve report directly at page 49, Figure 4
3. Try this que